In [ ]:
import numpy as np
import pandas as pd
from scipy import signal
from scipy import stats
from scipy import ndimage
import matplotlib.pyplot as plt
import importlib

from pxg import cbor
importlib.reload(cbor)

from pxg.cbor import CborDatabase, CborRecord

%config InlineBackend.figure_format = 'svg'

In [ ]:
### FILTERS ###

def LynnFilter(sig: np.ndarray, W: int) -> np.ndarray:
    W -= W % 2

    # cf = 0.3 * FS / (W/2)
    # print(cf)

    # Lynn filter
    a = np.array([1, -2, 1])
    b = np.zeros(W + 1)
    b[0] = 1
    b[W//2] = -2
    b[W] = 1
    g = (len(b) // 2) ** 2
    shift = W // 2 - 1

    sig = signal.lfilter(b, a, sig) / g # type: ignore
    sig = np.roll(sig, -shift)
    
    return sig
pass #def

def BaselineFilter(sig: np.ndarray, M: int) -> np.ndarray:
    if M <= 1: return sig
    # calculate moving median
    med = ndimage.median_filter(sig, size=M, mode="nearest")
    # med = np.convolve(med, np.ones(M)/M, mode="same")
    sig = sig - LynnFilter(med, W = M)
    return sig
pass #def

def SignalFilter(sig: np.ndarray, W: int, M: int, H: int = 0) -> np.ndarray:
    sig = BaselineFilter(sig, M)
    sig = LynnFilter(sig, W = W)
    if H > 0:
        hi = np.convolve(sig, np.ones(H)/H, mode="same")
        sig = sig - hi
    pass #if
    return sig
pass #def

In [ ]:
## FFT ##########################

# --- CONFIGURATION ---
FS = 250       # Sampling Rate (Hz)
FN = 1024      # Buffer Size (4.096 seconds)
MS = 4         # Tick size 4 ms (250 Hz)

WELCH_NPERSEG = 512     # Sub-window size for Welch (controls freq resolution vs. smoothing)
WELCH_NOVERLAP = 384    # 50% overlap is standard
WELCH_WINDOW = 'hann'   # Window function

BIN = FS / FN  # Frequency resolution (Hz per bin)
FREQ_LOW = int(2 / BIN)
FREQ_HIGH = int(15 / BIN)
FREQ_LIMIT = int(24.5 / BIN)

PEAK_HEIGHT = 0.01
PEAK_DIST = 3
HARM_TOLER = 0.5 # Hz tolerance for matching harmonics
FREQ_SPLIT = 5.0 # Hz split for band power ratio (e.g., LF/HF)
MEDI_PCTIL = 0.5 # Percentile for median-based prominence threshold
EDGE_PCTIL = 0.9 # Percentile for spectral edge frequency

class Spectral:
    def __init__(self, power: np.ndarray, freqs: np.ndarray | None = None):
        if freqs is None:
            # Apply Hann window to reduce leakage
            hann = np.hanning(FN)
            # hann = signal.windows.tukey(FN, alpha=0.5)
            tran = np.fft.rfft(power * hann)
            # Magnitude (Absolute value)
            # Normalization (Divide by sum of window)
            # This corrects for the energy loss of the windowing
            tran = np.abs(tran) / np.sum(hann)
            # Symmetry Compensation (Multiply by 2 for positive freqs)
            # We only keep the first half (0 to N/2)
            mags = tran * 2
            # Fix DC component (Index 0 should not be doubled)
            mags[0] = mags[0] / 2
            # Power is magnitude squared
            power = mags ** 2
            freqs = np.fft.rfftfreq(FN, 1/FS)
            
            power = power[:FREQ_LIMIT]
            freqs = freqs[:FREQ_LIMIT]


            # freqs, power = signal.welch(
            #     power,
            #     fs=FS,
            #     window=WELCH_WINDOW,
            #     nperseg=WELCH_NPERSEG,
            #     noverlap=WELCH_NOVERLAP,
            #     scaling='density',   # 'density' = V^2/Hz, 'spectrum' = V^2
            #     average='mean',      # or 'median' for robustness against artifacts
            # )
            # # Trim to desired frequency range
            # mask = freqs < 24.5
            # freqs = freqs[mask]
            # power = power[mask]
        pass #if

        self.Freqs = freqs
        self.Power = power
        self.Magnitude = np.sqrt(power)
        self.Total = np.sum(power)

        self.LFB = np.sum(power[(freqs > 0.5) & (freqs <= 3.5)])
        self.MFB = np.sum(power[(freqs > 3.5) & (freqs <= 8.0)])
        self.HFB = np.sum(power[(freqs > 8.0) & (freqs <= 20.0)])
        self.OFB = np.sum(power[(freqs > 1.5) & (freqs <= 24)])

        mask = (freqs > 1.66) & (freqs <= 8.33)
        self.DOM = freqs[mask][np.argmax(power[mask])]
        self.DFB = np.sum(power[(freqs > 1.5) & (freqs > self.DOM - 0.5) & (freqs < self.DOM + 0.5)])
        self.H2B = np.sum(power[(freqs > 1.5) & (freqs > self.DOM * 2 - 0.5) & (freqs < self.DOM * 2 + 0.5)])
        self.H3B = np.sum(power[(freqs > 1.5) & (freqs > self.DOM * 3 - 0.5) & (freqs < self.DOM * 3 + 0.5)])

        # find the second max peak ouside of the DOM +/- 0.5 Hz
        mask = (freqs > 1.5) & ((freqs < self.DOM - 0.5) | (freqs > self.DOM + 0.5)) & (freqs <= 24)
        self.D2M = freqs[mask][np.argmax(power[mask])]
        self.D2B = np.sum(power[(freqs > 1.5) & (freqs > self.D2M - 0.5) & (freqs < self.D2M + 0.5)])
        
        ### Normalize power for Entropy/Probabilistic calcs
        self.Normal = self.Power / self.Total if self.Total > 0 else self.Power

        ### Spectral Centroid (f_cent) is the "center of mass" of the spectrum
        if self.Total == 0: self.Total = 1e-10 # Avoid division by zero
        self.Centroid = np.sum(self.Freqs * self.Power) / self.Total

        ### Spectral Spread (f_spread) is the standard deviation of the spectrum around the centroid
        self.Spread = np.sqrt(np.sum(((self.Freqs - self.Centroid) ** 2) * self.Power) / self.Total)

        ### Spectral Entropy (H) measures the "disorder" of the spectrum
        # Filter out zeros to avoid log(0)
        p = self.Normal[self.Normal > 0]
        entropy = -np.sum(p * np.log2(p))
        # Normalize by log2 of number of bins (0 to 1 scale)
        self.Entropy = entropy / np.log2(len(self.Normal))

        ### Gini index measures inequality in the spectrum (0 to 1 scale)
        sorted = np.sort(power)
        idx = np.arange(n := len(sorted))
        multipliers = 2 * idx - n + 1
        gini_sum = np.dot(multipliers, sorted)
        self.Gini = gini_sum / (n * self.Total)
        
        ### Spectral Flatness (SF) measures how "noise-like" the spectrum is
        # Geometric Mean / Arithmetic Mean
        g_mean = stats.gmean(self.Power + 1e-10) # Add small epsilon to avoid log(0)
        a_mean = np.mean(self.Power) + 1e-10
        self.Flatness = g_mean / a_mean

        ### Edge Frequency (f_edge) is the frequency below which a certain percentage of total power is contained
        cumsum = np.cumsum(self.Normal)
        # Find index where cumsum crosses 0.95
        idx = np.searchsorted(cumsum, EDGE_PCTIL)
        # Handle edge case where index might be out of bounds
        idx = min(idx, len(self.Freqs) - 1)
        self.F90 = self.Freqs[idx]
        self.F50 = self.Freqs[np.searchsorted(cumsum, MEDI_PCTIL)]

        ### Dominant Frequency (f_dom) is the frequency with max power
        idx = np.argmax(self.Power)
        f_dom = self.Freqs[idx]
        self.Dominant = f_dom

        ### Peak Detection for Harmonics
        # dist is in 'bins'. Adjust based on your resolution.
        # peak_idxs, _ = signal.find_peaks(self.Normal, height=PEAK_HEIGHT, distance=PEAK_DIST)
        # self.Peaks = peak_idxs
        
        # Prominence must be at least 5% of the highest peak
        # This catches the small "parent" in a seesaw, but ignores tiny static
        mags = self.Magnitude
        min_prominence = 0.05 * np.max(mags)
        
        # Height must be above the average noise floor
        min_height = np.mean(mags) * 0.5

        # 2. The Core Detection
        peak_idxs, _ = signal.find_peaks(
            mags,
            prominence=min_prominence,
            distance=PEAK_DIST,  # ~1 Hz separation minimum
            # width=1.5,           # Reject 1-bin digital spikes
            height=min_height    # Reject absolute background static
        )
        self.Peaks = peak_idxs
        
        ### Band Power Ratio (BPR) is the ratio of power in a high-frequency band to a low-frequency band
        mask_low = self.Freqs <= FREQ_SPLIT
        mask_high = self.Freqs > FREQ_SPLIT
        
        p_low = np.sum(self.Power[mask_low])
        p_high = np.sum(self.Power[mask_high])
        
        if p_low == 0: 
            self.BPR = 10.0 # High value indicates HF dominance
        else:
            self.BPR = p_high / p_low
        pass #if

        # self.Power = np.convolve(self.Power, np.ones(5)/5, mode="same")
    pass #def
pass #class

In [ ]:
### SEGMENT ###

class Segment:
    def __init__(self, rec: CborRecord, start: int, end: int, epis: list[cbor.CborAnnotation], anns: list[cbor.CborAnnotation]):
        self.Epis = epis
        self.Anns = anns

        self.DB = rec.Info.DB
        self.RID = rec.Info.RID
        self.Start = start
        self.End = end

        self.Dominant = 0.00
        self.Centroid = 0.00
        self.Spread = 0.00
        self.Ginidex = 0.0000
        self.Entropy = 0.0000
        self.Flatness = 0.0000

        self.NSR = 0
        self.BGM = 0
        self.TGM = 0
        self.VTH = 0
        self.VFL = 0
        self.VFN = 0
        self.VFB = 0
        self.AFL = 0
        self.AFB = 0
        self.EPX = 0

        # // N: N, L, R, B
        self.N = 0
        self.L = 0
        self.R = 0
        self.B = 0
        # // A: A
        self.A = 0
        # // S: S
        self.S = 0
        # // C: J, a, j, e, n
        self.C = 0
        # // V: V, r, E
        self.V = 0
        self.W = 0
        # // F: F
        self.F = 0
        # // Q: Q, f
        self.Q = 0
        # // P: /
        self.P = 0
        # // O: ~, !, [, ]
        self.O = 0
        # // Z
        self.Z = 0
        # // Other
        self.X = 0

        for e in epis:
            a = max(start, e.Time)
            b = min(end, e.End)
            k = max(0, b - a)
            if e.Type == "[":
                self.VFN += k
            elif e.Note == "(N":
                self.NSR += k
            elif e.Note == "(B":
                self.BGM += k
            elif e.Note == "(T":
                self.TGM += k
            elif e.Note == "(VT":
                self.VTH += k
            elif e.Note == "(VFL":
                self.VFL += k
            elif e.Note == "(VF":
                self.VFB += k
            elif e.Note == "(AFL":
                self.AFL += k
            elif e.Note == "(AFIB":
                self.AFB += k
            else:
                self.EPX += k
            pass #if
        pass #for

        for a in anns:
            if a.Type == "N":
                self.N += 1
            elif a.Type == "L":
                self.L += 1
            elif a.Type == "R":
                self.R += 1
            elif a.Type == "B":
                self.B += 1
            elif a.Type == "A":
                self.A += 1
            elif a.Type == "S":
                self.S += 1
            elif a.Type in ["J", "a", "j", "e", "n"]:
                self.C += 1
            elif a.Type == "V":
                self.V += 1
            elif a.Type in ["r", "E"]:
                self.W += 1
            elif a.Type == "F":
                self.F += 1
            elif a.Type in ["Q", "f"]:
                self.Q += 1
            elif a.Type == "/":
                self.P += 1
            elif a.Type in ["~", "!", "[", "]"]:
                self.O += 1
            else:
                self.Z += 1
            pass #if
        pass #for

        sig = rec.Signal[start:end+24]
        if len(sig) < FN:
            sig = np.pad(sig, (0, FN - len(sig)), mode="constant")
        pass #if
        spr = Spectral(sig)

        self.DOM = spr.DOM
        self.DFB = spr.DFB
        self.H2B = spr.H2B
        self.H3B = spr.H3B

        self.D2M = spr.D2M
        self.D2B = spr.D2B
        
        self.LFB = spr.LFB
        self.MFB = spr.MFB
        self.HFB = spr.HFB
        self.OFB = spr.OFB

        self.F50 = spr.F50
        self.F90 = spr.F90

        self.Dominant = spr.Dominant
        self.Centroid = spr.Centroid
        self.Spread = spr.Spread
        self.Ginidex = spr.Gini
        self.Entropy = spr.Entropy
        self.Flatness = spr.Flatness
    pass #def

    def Line(self):
        return f"{self.DB}\t{self.RID}\t{self.Start}\t{self.End}\t" + \
               f"{self.DOM:.2f}\t{self.DFB:.4f}\t{self.H2B:.4f}\t{self.H3B:.4f}\t" + f"{self.D2M:.2f}\t{self.D2B:.4f}\t" + \
               f"{self.LFB:.4f}\t{self.MFB:.4f}\t{self.HFB:.4f}\t{self.OFB:.4f}\t" + f"{self.F50:.2f}\t{self.F90:.2f}\t" + \
               f"{self.Dominant:.2f}\t{self.Centroid:.2f}\t{self.Spread:.4f}\t{self.Ginidex:.4f}\t{self.Entropy:.4f}\t{self.Flatness:.4f}\t" + \
               f"{self.NSR}\t{self.BGM}\t{self.TGM}\t{self.VTH}\t{self.VFL}\t{self.VFN}\t{self.VFB}\t{self.AFL}\t{self.AFB}\t{self.EPX}\t" + \
               f"{self.N}\t{self.L}\t{self.R}\t{self.B}\t{self.A}\t{self.S}\t{self.C}\t{self.V}\t{self.W}\t{self.F}\t{self.Q}\t{self.P}\t{self.O}\t{self.Z}"
    pass #def
pass #class

HEADER = "DB\tRID\tStart\tEnd\tDOM\tDFB\tH2B\tH3B\tD2M\tD2B\tLFB\tMFB\tHFB\tOFB\tF50\tF90\tDominant\tCentroid\tSpread\tGinidex\tEntropy\tFlatness\tNSR\tBGM\tTGM\tVTH\tVFL\tVFN\tVFB\tAFL\tAFB\tEPX\tN\tL\tR\tB\tA\tS\tC\tV\tW\tF\tQ\tP\tO\tZ"

LYN_WIND = 48 // MS
MED_WIND = 596 // MS

def Process(rec: CborRecord, out, STEP = 250):
    rec.Signal = SignalFilter(rec.Signal, W = LYN_WIND, M = MED_WIND, H = 0)
    anns = rec.Annotations
    size = len(rec.Signal) # rec.Info.Channels[rec.Best].Size
    
    aix = 0
    epis = []
    beat = []
    for t in range(STEP, size, STEP):
        while aix < len(anns) and anns[aix].Time < t:
            if anns[aix].Type == "[" or anns[aix].Type == "+" and anns[aix].Note[:1] == "(":
                epis.append(anns[aix])
            else:
                beat.append(anns[aix])
            pass #if
            aix += 1
        pass #while
        c = t - 1000
        r = 0
        while r < len(beat) and beat[r].Time < c:
            r += 1
        pass #while
        beat = beat[r:]
        r = 0
        while r < len(epis) and epis[r].End <= c:
            r += 1
        pass #while
        epis = epis[r:]

        if c < 0: continue

        seg = Segment(rec, c, t, epis, beat)
        out.write(seg.Line())
        out.write("\n")
    pass #for
pass #def

def RunDatabase(DB: str, STEP = 250):
    cdb = CborDatabase(DB, freq=250, chn=0)

    out = open(f"../work/output/fft/{DB}.fft.tsv", "w")
    out.write(HEADER + "\n")

    for rid in cdb.Records:
        rec = cdb.Load(rid)

        print(">>", rid)
        Process(rec, out, STEP)
        continue

        rec.Annotations = [a for a in rec.Annotations if a.Type in ["+", "[", "]", "~"]]
        print(rec.Info, rec.Info.Channels[rec.Best])
        for a in rec.Annotations:
            print(a.Time, a.Type, a.Note, a.End)
        pass #for
    pass #for
pass #for

In [ ]:
def PlotRecord(rf: pd.DataFrame, rid: str):
    plt.figure(figsize=(24, 3))
    plt.title(f"{rid}")
    plt.ylim(-6, 12)

    t = rf["Start"] / 250
    plt.xlim(0, t.max())

    plt.plot(t, rf["DOM"], color="green", alpha=0.5)
    plt.plot(t, rf["Dominant"], color="green", alpha=0.5, linewidth=0.5, linestyle="--")

    plt.plot(t, rf["Centroid"], color="blue", alpha=0.5, linewidth=0.7)
    # plt.plot(t, rf["Spread"], color="red", alpha=0.5, linewidth=0.5, linestyle="--")

    plt.axhspan(10, 12, color="gray", alpha=0.1)
    plt.plot(t, rf["Entropy"] * 4 + 10 - 2, color="black", alpha=0.7, linewidth=0.7)
    # R = rf["MFB"] / (rf["HFB"] + rf["MFB"] + 1e-10)
    # plt.plot(t, R * 10, color="black", alpha=0.7)

    base = -6
    mul = 4
    WFB = (rf["LFB"] + rf["MFB"] + rf["HFB"]) / mul
    # plot stacked area for LFB/MFB/HFB
    plt.fill_between(t, base + 0, base + rf["LFB"] / WFB, color="gray", alpha=0.3)
    plt.fill_between(t, base + rf["LFB"] / WFB, base + (rf["LFB"] + rf["MFB"]) / WFB, color="blue", alpha=0.3)
    plt.fill_between(t, base + (rf["LFB"] + rf["MFB"]) / WFB, base + mul, color="purple", alpha=0.3)
    plt.plot(t, rf["DFB"] / rf["OFB"] * mul + base, color="magenta", alpha=0.7, linewidth=0.7)

    base = -2
    mul = 2
    plt.axhspan(base, 0, color="gray", alpha=0.1)
    vth = rf["VTH"] / 1000 * mul + base
    vfl = rf["VFL"] / 1000 * mul + base
    vfn = rf["VFN"] / 1000 * mul + base
    vfb = rf["VFB"] / 1000 * mul + base

    # plt.plot(t, vth, color="green", alpha=0.7)
    plt.fill_between(t, base, vth, color="green", alpha=0.15)

    # plt.plot(t, vfl, color="orange", alpha=0.7)
    plt.fill_between(t, base, vfl, color="orange", alpha=0.2)

    # plt.plot(t, vfn, color="orange", alpha=0.7)
    plt.fill_between(t, base, vfn, color="orange", alpha=0.2)

    # plt.plot(t, vfb, color="magenta", alpha=0.7)
    plt.fill_between(t, base, vfb, color="magenta", alpha=0.15)

    plt.show()
pass #def

def PlotDatabase(DB: str):
    df = pd.read_csv(f"../work/fft/{DB}.fft.tsv", sep="\t", dtype={"RID": str})

    cdb = CborDatabase(DB, freq=250, chn=0)
    for rid in cdb.Records:
        PlotRecord(df[df["RID"] == rid], rid)
    pass #for
pass #def

In [ ]:
# RunDatabase("ahadb")

# RunDatabase("mitdb")
# RunDatabase("cudb")
# RunDatabase("vfdb")

# PlotDatabase("ahadb")
# PlotDatabase("cudb")
# PlotDatabase("vfdb")


# DB	RID	Start	End	DOM	DFB	HRB	LFB	MFB	HFB	OFB	Dominant	Centroid	Spread	Ginidex	Entropy	Flatness	NSR	BGM	TGM	VTH	VFL	VFN	VFB	AFL	AFB	EPX	


def LoadDataset(DB: str) -> pd.DataFrame:
    df = pd.read_csv(f"../work/output/fft/{DB}.fft.tsv", sep="\t", dtype={"RID": str})
    df["Target"] = "MIX"
    for idx, row in df.iterrows():
        if row["VTH"] + row["VFL"] + row["VFN"] + row["VFB"] + row["AFL"] + row["AFB"]  + row["BGM"]*0 + row["TGM"]*0 == 0: 
            df.at[idx, "Target"] = "NSR"
        elif row["VTH"] > 900:
            df.at[idx, "Target"] = "VTH"
        elif row["VFL"] > 900:
            df.at[idx, "Target"] = "VFL"
        elif row["VFN"] > 900:
            df.at[idx, "Target"] = "VFL"
        elif row["VFB"] > 900:
            df.at[idx, "Target"] = "VFB"
        else:
            df.at[idx, "Target"] = "MIX"
        pass #if
        continue

        if row["VTH"] > 500:
            df.at[idx, "Target"] = "VTH"
        elif row["VFL"] > 500:
            df.at[idx, "Target"] = "VFL"
        elif row["VFN"] > 500:
            df.at[idx, "Target"] = "VFL"
        elif row["VFB"] > 500:
            df.at[idx, "Target"] = "VFB"
        elif row["AFL"] > 500:
            df.at[idx, "Target"] = "NSR"
        elif row["AFB"] > 500:
            df.at[idx, "Target"] = "NSR"
        else:
            df.at[idx, "Target"] = "NSR"
        pass #if
    pass #for
    # print count per Target
    print(df["Target"].value_counts())

    fb = df["LFB"] + df["MFB"] + df["HFB"] + 1e-10
    
    df["MXH"] = df["MFB"] / (df["HFB"] + 1e-10)
    df["MXL"] = df["MFB"] / (df["LFB"] + 1e-10)
    df["LXH"] = df["LFB"] / (df["HFB"] + 1e-10)
    # df["LXM"] = df["LFB"] / (df["MFB"] + 1e-10)
    # df["HXL"] = df["HFB"] / (df["LFB"] + 1e-10)
    # df["HXM"] = df["HFB"] / (df["MFB"] + 1e-10)

    df["LFB"] = df["LFB"] / fb
    df["MFB"] = df["MFB"] / fb
    df["HFB"] = df["HFB"] / fb

    df["D2B"] = df["D2B"] / (df["DFB"] + 1e-10)
    
    df["H2B"] = df["H2B"] / (df["DFB"] + 1e-10)
    df["H3B"] = df["H3B"] / (df["DFB"] + 1e-10)
    df["DFB"] = df["DFB"] / (df["OFB"] + 1e-10)

    return df[df["Target"] != "MIX"].reset_index(drop=True)
pass #def

# RunDatabase("mitdb", STEP = 250)
# RunDatabase("cudb", STEP = 250)
# RunDatabase("vfdb", STEP = 250)
# RunDatabase("ahadb", STEP = 250)

PlotDatabase("mitdb")

# df = LoadDataset("vfdb")

ahadf = LoadDataset("ahadb")
mitdf = LoadDataset("mitdb")
cudf = LoadDataset("cudb")
vfdf = LoadDataset("vfdb")
rdf = pd.concat([ahadf, mitdf, cudf, vfdf], ignore_index=True)
print(rdf["Target"].value_counts())


In [ ]:
# plot a histogram of the DOM for each Target
targets = ["NSR", "VTH", "VFL", "VFB"]
bins = 50

def PlotHist(var: str):
    _, axes = plt.subplots(1, len(targets), figsize=(12, 3), sharex=True, sharey=True)
    dom_min = rdf[var].min()
    dom_max = rdf[var].quantile(0.995)

    for ax, target in zip(axes, targets):
        x = rdf.loc[rdf["Target"] == target, var]
        # if len(x) > 0:
        #     ax.hist(x, bins=bins, range=(dom_min, dom_max), density=True, alpha=0.7)
        ax.hist(x, bins=bins, range=(dom_min, dom_max), weights=np.ones(len(x))/len(x), alpha=0.7)
        ax.set_title(f"{target} (n={len(x)})")
        # ax.set_xlabel(var)
    pass #for

    axes[0].set_ylabel(var)
    plt.tight_layout()
    plt.show()
pass #def

PlotHist("Centroid")
PlotHist("Spread")
PlotHist("Ginidex")
PlotHist("Entropy")

PlotHist("DOM")
PlotHist("DFB")
PlotHist("D2M")
PlotHist("D2B")

PlotHist("H2B")
PlotHist("H3B")

PlotHist("LFB")
PlotHist("MFB")
PlotHist("HFB")

PlotHist("MXH")
PlotHist("MXL")
PlotHist("LXH")




In [ ]:
# train decision tree on Target using DOM, DFB, HRB, LFB, MFB, HFB, Centroid, Spread, Ginidex, Entropy, Flatness
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# fb = df["LFB"] + df["MFB"] + df["HFB"] + 1e-10
# df["MFX"] = df["MFB"] / (df["HFB"] + 1e-10)
# df["LFB"] = df["LFB"] / fb
# df["MFB"] = df["MFB"] / fb
# df["HFB"] = df["HFB"] / fb

# df["HRB"] = df["HRB"] / (df["DFB"] + 1e-10)
# df["DFB"] = df["DFB"] / (df["OFB"] + 1e-10)

# features = [ "HRB" ] 
features = [
    "DOM", # Dominant frequency (Hz)
    "DFB", # +/- 0.5 Hz around DOM / Out-of-band power (1.5-24 Hz)
    "H2B", # +/- 0.5 Hz around (DOM * 2) / DFB
    "H3B", # +/- 0.5 Hz around (DOM * 3) / DFB

    "D2M", # Second peak frequency (Hz)
    "D2B", # +/- 0.5 Hz around D2M / DFB

    "LFB", # Low-frequency band power (0.5-3.5 Hz) / (LFB + MFB + HFB)
    "MFB", # Mid-frequency band power (3.5-8 Hz) / (LFB + MFB + HFB)
    "HFB", # High-frequency band power (8-20 Hz) / (LFB + MFB + HFB)

    "F50", # Spectral edge frequency 50% (Hz)
    "F90", # Spectral edge frequency 90% (Hz)
    
    "MXH", # Mid-frequency dominance ratio (MFB / HFB)
    "MXL", # Mid-frequency dominance ratio (MFB / LFB)
    "LXH", # Low-frequency dominance ratio (LFB / HFB)

    "Centroid", # Centorid of the spectrum (Hz) = SUM(Freq * Power) / SUM(Power)
    "Spread",   # Spread of the spectrum (Hz) = SQRT(SUM((Freq - Centroid)^2 * Power) / SUM(Power))
    "Ginidex",  # Gini index of the spectrum
    "Entropy",  # Shannon entropy of the spectrum
    "Flatness", # Spectral flatness
]

df = rdf.copy()

# avoid chained assignment and keep original df unchanged
# df = df.copy()
# df.loc[df["Target"] != "VFL", "Target"] = "NON"
# print(df["Target"].value_counts())

X = df[features]
y = df["Target"]

# 1) Proper split (stratified)
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )
X_train, X_test, y_train, y_test = X, X, y, y

# 2) Class weights
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))
print("Weights:", class_weight)

clf = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=20,
    random_state=42,
    class_weight=class_weight
)
clf.fit(X_train, y_train)

# print sorted feature importance
importances = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=False)
print("Feature importance:")
print(importances.to_string(float_format=lambda v: f"{v:.6f}"))
print()

y_pred = clf.predict(X_test)
# print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred, labels=classes)
display(pd.DataFrame(cm, index=classes, columns=classes))
print(classification_report(y_test, y_pred, digits=4))


# plot the decision tree
from sklearn.tree import plot_tree
plt.figure(figsize=(15, 8))
plot_tree(clf, feature_names=features, class_names=clf.classes_, filled=True, rounded=True, max_depth=3) #type: ignore
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

# ─────────────────────────────────────────
# 1. Taylor-Fourier Transform core function
# ─────────────────────────────────────────

def taylor_fourier_transform(signal, fs, freqs, L=2):
    """
    Compute Taylor-Fourier Transform.
    
    Parameters:
        signal : 1D array — ECG segment
        fs     : sampling frequency (Hz)
        freqs  : list of analysis frequencies (Hz)
        L      : Taylor expansion order (0=standard DFT, 1=linear, 2=quadratic)
    
    Returns:
        coeffs : complex array, shape (len(freqs), L+1)
                 coeffs[k, l] = l-th Taylor coefficient at freqs[k]
        t      : normalized time vector
    """
    N = len(signal)
    t = np.arange(N) / fs
    t_norm = t - t[N//2]  # center time around zero
    
    # Build Taylor-Fourier matrix H
    # Each row = one sample, each column = one basis function
    cols = []
    for f in freqs:
        for l in range(L + 1):
            # l-th Taylor term at frequency f
            basis = (t_norm ** l) * np.exp(1j * 2 * np.pi * f * t)
            cols.append(basis)
    
    H = np.column_stack(cols)  # shape: (N, K*(L+1))
    
    # Least squares solution: c = pinv(H) @ signal
    coeffs_flat, _, _, _ = np.linalg.lstsq(H, signal, rcond=None)
    
    # Reshape to (n_freqs, L+1)
    coeffs = coeffs_flat.reshape(len(freqs), L + 1)
    
    return coeffs, t_norm


# ─────────────────────────────────────────
# 2. Extract features from TFT coefficients
# ─────────────────────────────────────────

def tft_features(coeffs, freqs):
    """
    Extract amplitude, phase, and instantaneous frequency from TFT coefficients.
    
    Returns dict of features per frequency component.
    """
    results = []
    for k, f in enumerate(freqs):
        c0 = coeffs[k, 0]   # zeroth order — main phasor
        c1 = coeffs[k, 1] if coeffs.shape[1] > 1 else 0  # first order
        
        amplitude = np.abs(c0)
        phase     = np.angle(c0)
        
        # Instantaneous frequency correction from first-order coefficient
        # df = Im(c1/c0) / (2π) — frequency deviation from nominal
        if amplitude > 1e-10:
            freq_correction = np.imag(c1 / c0) / (2 * np.pi)
        else:
            freq_correction = 0
            
        inst_freq = f + freq_correction
        
        results.append({
            'nominal_freq': f,
            'amplitude':    amplitude,
            'phase':        phase,
            'inst_freq':    inst_freq,
            'power':        amplitude ** 2,
        })
    
    return results


# ─────────────────────────────────────────
# 3. Sliding window TFT for time evolution
# ─────────────────────────────────────────

def sliding_tft(signal, fs, freqs, window_sec=4.0, step_sec=0.5, L=2):
    """
    Apply TFT in a sliding window to track spectral evolution over time.
    
    Returns:
        times      : center time of each window
        amplitudes : array (n_windows, n_freqs)
        inst_freqs : array (n_windows, n_freqs)
    """
    N_win  = int(window_sec * fs)
    N_step = int(step_sec * fs)
    N      = len(signal)
    
    times      = []
    amplitudes = []
    inst_freqs = []
    
    start = 0
    while start + N_win <= N:
        segment = signal[start : start + N_win]
        coeffs, _ = taylor_fourier_transform(segment, fs, freqs, L=L)
        features  = tft_features(coeffs, freqs)
        
        times.append((start + N_win // 2) / fs)
        amplitudes.append([f['amplitude'] for f in features])
        inst_freqs.append([f['inst_freq']  for f in features])
        
        start += N_step
    
    return (np.array(times),
            np.array(amplitudes),
            np.array(inst_freqs))


# ─────────────────────────────────────────
# 4. Demo — simulate VFib-like signal
# ─────────────────────────────────────────

def simulate_vfib(duration=10, fs=250):
    """Simulate a coarse-to-fine VFib-like signal with drifting frequency."""
    t = np.arange(int(duration * fs)) / fs
    
    # Dominant frequency drifts from 7 Hz down to 4 Hz (coarse → fine VFib)
    freq_drift = 7 - 3 * (t / duration)
    phase      = 2 * np.pi * np.cumsum(freq_drift) / fs
    
    # Amplitude decays over time
    amplitude  = 1.0 * np.exp(-0.15 * t)
    
    # Add harmonics and noise
    signal = (amplitude * np.sin(phase) +
              0.3 * amplitude * np.sin(2 * phase + 0.5) +
              0.15 * np.random.randn(len(t)))
    
    return t, signal


# ─────────────────────────────────────────
# 5. Run and visualize
# ─────────────────────────────────────────

fs = 250
t, vfib = simulate_vfib(duration=10, fs=fs)

# Bandpass filter first
b, a = butter(4, [1/(fs/2), 40/(fs/2)], btype='band')
vfib_filt = filtfilt(b, a, vfib)

# Define analysis frequencies — harmonics of expected VFib range
analysis_freqs = np.arange(2, 20, 1.0)  # 2 to 19 Hz, 1 Hz spacing

# Run sliding TFT
times, amplitudes, inst_freqs = sliding_tft(
    vfib_filt, fs,
    freqs=analysis_freqs,
    window_sec=4.0,
    step_sec=0.25,
    L=2
)

# ── Figure 1: Raw signal ──────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 12))
fig.suptitle('Taylor-Fourier Transform — VFib Analysis', fontsize=14, fontweight='bold')

axes[0].plot(t, vfib_filt, color='steelblue', lw=0.8)
axes[0].set_title('Filtered ECG (simulated VFib)')
axes[0].set_ylabel('Amplitude')
axes[0].set_xlabel('Time (s)')
axes[0].grid(True, alpha=0.3)

# ── Figure 2: TFT Spectrogram (amplitude over time per frequency) ──
img = axes[1].pcolormesh(times, analysis_freqs, amplitudes.T,
                          shading='auto', cmap='hot')
axes[1].set_title('TFT Amplitude Spectrogram')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_xlabel('Time (s)')
plt.colorbar(img, ax=axes[1], label='Amplitude')

# ── Figure 3: Instantaneous frequency of dominant mode ──────────────
# Find dominant frequency at each time step
dom_idx  = np.argmax(amplitudes, axis=1)
dom_freq = inst_freqs[np.arange(len(times)), dom_idx]

axes[2].plot(times, dom_freq, color='crimson', lw=1.5, label='Dominant IF')
axes[2].axhline(9, color='gray', linestyle='--', alpha=0.5, label='Flutter upper bound')
axes[2].axhline(4, color='gray', linestyle=':',  alpha=0.5, label='VFib lower bound')
axes[2].set_title('Instantaneous Dominant Frequency over Time')
axes[2].set_ylabel('Frequency (Hz)')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylim(0, 15)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# ── Figure 4: Amplitude of dominant mode (VFib coarseness) ──────────
dom_amp = amplitudes[np.arange(len(times)), dom_idx]

axes[3].plot(times, dom_amp, color='darkorange', lw=1.5)
axes[3].set_title('Dominant Mode Amplitude over Time (coarse → fine VFib)')
axes[3].set_ylabel('Amplitude')
axes[3].set_xlabel('Time (s)')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
# plt.savefig('/mnt/user-data/outputs/tft_vfib_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Done")